In [4]:
import ee
import geemap
ee.Initialize(project='mineral-prospectivity-zim')
ROI = ee.Geometry.Rectangle([25.24, -22.42, 33.07, -15.61])

# Initialize with the project that owns your asset


# Load your dry-season composite
dry = ee.Image('projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_dry_season')

# Find the 373K hot pixel (fast)
max_val = dry.select('TIR').reduceRegion(
    reducer=ee.Reducer.max(),
    scale=300,
    maxPixels=1e9,
    bestEffort=True
).getInfo()['TIR']

mask = dry.select('TIR').eq(max_val)
coords = ee.Image.pixelLonLat().updateMask(mask).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=ROI,
    scale=300,
    maxPixels=1e9,
    bestEffort=True
).getInfo()

lon, lat = coords['longitude'], coords['latitude']
print(f"Hotspot at {lat:.5f}, {lon:.5f} | {max_val:.2f} K")

# Show map directly on that point
Map = geemap.Map(center=[lat, lon], zoom=12)
Map.add_basemap('Esri.WorldImagery')
Map.addLayer(dry.select(['Red','Green','Blue']), {'min':0,'max':0.4,'gamma':1.2}, 'True Color')
Map.addLayer(dry.select('TIR'), {'min':280,'max':373,'palette':['blue','cyan','yellow','red'], 'opacity':0.7}, 'Thermal')
Map

Hotspot at -18.79051, 29.78589 | 373.00 K


Map(center=[-18.79050995557009, 29.785889033193037], controls=(WidgetControl(options=['position', 'transparent…

In [5]:
composite = ee.Image('projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_dry_season')
#checks if the data on the landsat dry season dataset was properly masked .

stats = composite.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=ROI,
    scale=300,
    maxPixels=1e9,
    bestEffort=True,
    tileScale=4
)
print(stats.getInfo())

{'Blue_max': 0.38489771484375, 'Blue_mean': 0.053681155789257494, 'Blue_min': -0.011250312500000003, 'Green_max': 0.464643203125, 'Green_mean': 0.0835286146778486, 'Green_min': 0.008437324218749997, 'NIR_max': 0.57582140625, 'NIR_mean': 0.22993252448417315, 'NIR_min': -0.0018536770833333372, 'Red_max': 0.51119017578125, 'Red_mean': 0.11396727864593026, 'Red_min': 0.0033835144779512057, 'SWIR1_max': 0.64331564453125, 'SWIR1_mean': 0.29164785001383586, 'SWIR1_min': 0.0013967039015997033, 'SWIR2_max': 0.538229601531498, 'SWIR2_mean': 0.20752248468846998, 'SWIR2_min': 0.002062993164062495, 'TIR_max': 372.9999407000001, 'TIR_mean': 305.39759702166793, 'TIR_min': 281.85957674}


In [6]:
composite = ee.Image('projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_median_2019_2024')
#checks if the mask worked on landsat median.

stats = composite.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=ROI,
    scale=300,
    maxPixels=1e9,
    bestEffort=True,
    tileScale=4
)
print(stats.getInfo())

{'Blue_max': 0.3688571638299851, 'Blue_mean': 0.05019332045978705, 'Blue_min': -0.021475657552083336, 'Green_max': 0.4367332060546875, 'Green_mean': 0.08154504335126392, 'Green_min': 0.010315975260416662, 'NIR_max': 0.5530554847005209, 'NIR_mean': 0.24168030732238782, 'NIR_min': -0.0011921422830930219, 'Red_max': 0.4779171555834573, 'Red_mean': 0.10518624667261803, 'Red_min': 0.0031359084909916, 'SWIR1_max': 0.6310993104784124, 'SWIR1_mean': 0.28044275328127427, 'SWIR1_min': 0.0018667819923199522, 'SWIR2_max': 0.535161606593541, 'SWIR2_mean': 0.19479939974273403, 'SWIR2_min': 0.002394161483759162, 'TIR_max': 372.9999407000001, 'TIR_mean': 307.4581687221098, 'TIR_min': 246.95361716}


In [1]:
import ee
ee.Initialize(project='mineral-prospectivity-zim')

assets = ee.data.listAssets('projects/mineral-prospectivity-zim/assets')
for a in assets['assets']:
    print(a['id'], a['type'])

projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_dry_season IMAGE
projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_median_2019_2024 IMAGE
